# 05 - Model Evaluation
**Objective**: Deep-dive into the best model's performance with ROC curves, confusion matrices, and classification reports.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report
from src.utils.config import PROCESSED_DATA_DIR, MODELS_DIR
from src.utils.helpers import load_dataframe, load_model
from src.visualization.plots import plot_roc_curves, plot_confusion_matrix
from src.models.evaluator import create_comparison_table
from src.features.selection import get_feature_columns
from src.models.trainer import prepare_train_test

## 5.1 Load Artifacts

In [ ]:
# Load engineered data and training artifacts
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'))
artifacts = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'training_artifacts.pkl'))

selected_features = artifacts['selected_features']
best_name = artifacts['best_model_name']
scaler = artifacts['scaler']

# Re-prepare train/test with same split
X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, _ = \
    prepare_train_test(df, selected_features)

# Load all saved models and re-predict
from src.models.trainer import get_models
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)

models_dict = {}
results = {}
for name in ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']:
    safe = name.lower().replace(' ', '_')
    model = load_model(os.path.join(MODELS_DIR, f'{safe}.pkl'))
    models_dict[name] = model

    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'AUC-PR': average_precision_score(y_test, y_proba),
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
    }

## 5.2 Model Comparison Table

In [ ]:
comparison = create_comparison_table(results)
display(comparison.style.highlight_max(axis=0, color='lightgreen'))

## 5.3 ROC & Precision-Recall Curves

In [ ]:
plot_roc_curves(results, y_test)

## 5.4 Confusion Matrix (Best Model)

In [ ]:
best_res = results[best_name]
plot_confusion_matrix(y_test, best_res['y_pred'], best_name)

print(f"\nClassification Report - {best_name}")
print(classification_report(y_test, best_res['y_pred'], target_names=['Saved', 'Lost']))